# Polynomial ARX Notebook (Greenhouse)

Mục tiêu:
- Tạo dữ liệu nhà kính có phi tuyến bằng file mới `data_generator_poly_arx.py`.
- So sánh ARX tuyến tính và Polynomial ARX trên cùng dữ liệu.
- Chứng minh bằng 2 chỉ số chính: `FIT_12step` và `FIT_free_run`.


In [1]:
import numpy as np
import pandas as pd

from arx_pipeline import (
    SplitConfig,
    ModelConfig,
    split_time_series,
    build_regression_matrix,
    estimate_ols,
    evaluate_slice,
)
from narx_pipeline import (
    NARXModelConfig,
    build_narx_regression_matrix,
    evaluate_narx_slice,
)
from data_generator_poly_arx import generate_greenhouse_data_poly


## 1) Sinh dữ liệu phi tuyến (file mới, không đụng dữ liệu cũ)


In [2]:
df_raw, true_params = generate_greenhouse_data_poly(days=365, sampling_seconds=300, seed=42)
df_raw.to_csv('greenhouse_data_poly.csv', index=False)
print(f'Rows: {len(df_raw)}')
print('Saved: greenhouse_data_poly.csv')
print('True nonlinear params:', {k: true_params[k] for k in ['c_temp2', 'c_humi2', 'c_temp_humi']})
df_raw.head()


Rows: 105120
Saved: greenhouse_data_poly.csv
True nonlinear params: {'c_temp2': 0.9, 'c_humi2': -0.5, 'c_temp_humi': 0.7}


,Timestamp,Month,Season,Soil_Moisture,Temperature,Humidity,Light,Drip,Mist,Fan
0,2025-01-01 00:00:00,1,winter,58.000000,21.383673,78.919777,5.679396,0.0,0.0,0.0
1,2025-01-01 00:05:00,1,winter,58.000000,20.415301,76.340090,19.224024,0.0,0.0,0.0
2,2025-01-01 00:10:00,1,winter,56.181273,21.643836,77.756090,-7.088395,1.0,1.0,0.0
3,2025-01-01 00:15:00,1,winter,54.653405,21.754469,79.074277,17.759743,0.0,0.0,0.0
4,2025-01-01 00:20:00,1,winter,53.535424,19.710237,79.514494,-11.489911,0.0,1.0,1.0


## 2) Min-Max normalization (đúng quy trình Polynomial ARX)


In [3]:
df = df_raw.copy()
cols = ['Soil_Moisture', 'Temperature', 'Humidity', 'Light', 'Drip', 'Mist', 'Fan']
minmax = {}
for c in cols:
    cmin = float(df[c].min())
    cmax = float(df[c].max())
    minmax[c] = (cmin, cmax)
    denom = cmax - cmin if (cmax - cmin) != 0 else 1.0
    df[c] = (df[c] - cmin) / denom

split_cfg = SplitConfig(train_ratio=0.60, val_ratio=0.20)
df_train, df_val, df_test = split_time_series(df, split_cfg)
print('Train/Val/Test:', len(df_train), len(df_val), len(df_test))


Train/Val/Test: 63072 21024 21024


## 3) ARX baseline


In [4]:
arx_cfg = ModelConfig(
    na=2,
    nb=2,
    nk=1,
    include_intercept=False,
    simulation_clip=(0.0, 1.0),
)
X_arx, y_arx = build_regression_matrix(df_train, arx_cfg)
theta_arx, _, _ = estimate_ols(X_arx, y_arx)
val_arx = evaluate_slice('Validation', df_val, theta_arx, arx_cfg, n_step=12)
test_arx = evaluate_slice('Test', df_test, theta_arx, arx_cfg, n_step=12)


## 4) Polynomial ARX (bậc 2)
Dùng thêm bình phương và tích chéo thông qua ma trận hồi quy polynomial.


In [5]:
poly_cfg = NARXModelConfig(
    na=2,
    nb=2,
    nk=1,
    poly_degree=2,
    include_intercept=True,
    cross_term_mode='all',
    simulation_clip=(0.0, 1.0),
)
X_poly, y_poly = build_narx_regression_matrix(df_train, poly_cfg)
theta_poly, _, _ = estimate_ols(X_poly, y_poly)
val_poly = evaluate_narx_slice('Validation', df_val, theta_poly, poly_cfg, n_step=12, arx_theta=theta_arx)
test_poly = evaluate_narx_slice('Test', df_test, theta_poly, poly_cfg, n_step=12, arx_theta=theta_arx)


## 5) So sánh trọng tâm: FIT_12step và FIT_free_run


In [6]:
compare = pd.DataFrame([
    {
        'Split': 'Validation',
        'ARX_FIT_12step': val_arx['metrics_n_step']['FIT'],
        'PolyARX_FIT_12step': val_poly['metrics_n_step']['FIT'],
        'ARX_FIT_free_run': val_arx['metrics_sim']['FIT'],
        'PolyARX_FIT_free_run': val_poly['metrics_sim']['FIT'],
    },
    {
        'Split': 'Test',
        'ARX_FIT_12step': test_arx['metrics_n_step']['FIT'],
        'PolyARX_FIT_12step': test_poly['metrics_n_step']['FIT'],
        'ARX_FIT_free_run': test_arx['metrics_sim']['FIT'],
        'PolyARX_FIT_free_run': test_poly['metrics_sim']['FIT'],
    },
])
compare['Delta_12step'] = compare['PolyARX_FIT_12step'] - compare['ARX_FIT_12step']
compare['Delta_free_run'] = compare['PolyARX_FIT_free_run'] - compare['ARX_FIT_free_run']
display(compare.round(4))

val_ok = (compare.loc[compare['Split']=='Validation', 'Delta_12step'].iloc[0] > 0) and (compare.loc[compare['Split']=='Validation', 'Delta_free_run'].iloc[0] > 0)
test_ok = (compare.loc[compare['Split']=='Test', 'Delta_12step'].iloc[0] > 0) and (compare.loc[compare['Split']=='Test', 'Delta_free_run'].iloc[0] > 0)

print('Validation improved on both metrics:', val_ok)
print('Test improved on both metrics:', test_ok)
if val_ok and test_ok:
    print('KET LUAN: Polynomial ARX tot hon ARX tren ca FIT_12step va FIT_free_run.')
else:
    print('KET LUAN: Can dieu chinh cau hinh polynomial de cai thien dong thoi ca 2 chi so.')


,Split,ARX_FIT_12step,PolyARX_FIT_12step,ARX_FIT_free_run,PolyARX_FIT_free_run,Delta_12step,Delta_free_run
0,Validation,28.5218,47.5720,27.8823,47.5122,19.0502,19.6299
1,Test,29.1958,49.0586,28.2228,48.9577,19.8628,20.7348


Validation improved on both metrics: True
Test improved on both metrics: True
KET LUAN: Polynomial ARX tot hon ARX tren ca FIT_12step va FIT_free_run.
